**Welcome to Weekly-Assignment 4 on Deep Learning for Computer Vision.**

This assignment is based on the content you learned in Week-4 of course.


#### **Instructions**
1. Use Python 3.x to run this notebook
2. Write your code only in between the lines 'YOUR CODE STARTS HERE' and 'YOUR CODE ENDS HERE'.
you should not change anything else in the code cells, if you do, the answers you are supposed to get at the end of this assignment might be wrong.
3. Read documentation of each function carefully.
4. All the Best!

##MNIST classification using CNN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torch.nn.functional as F
import timeit
import unittest

## Please DONOT remove these lines.
torch.manual_seed(2024)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(2024)

### Data Loading and Pre-processing

In [ ]:
# check availability of GPU and set the device accordingly
#### YOUR CODE STARTS HERE ####
device =torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
#### YOUR CODE ENDS HERE ####

# Hyper parameters
num_epochs = 10
num_classes = 10
learning_rate = 0.01

# define a transforms for preparing the dataset
# for normalization of the MNIST dataset, take mean=0.1307 and std=0.3081

#### YOUR CODE STARTS HERE ####

#### YOUR CODE ENDS HERE ####

In [ ]:
# Load the MNIST training, test datasets using `torchvision.datasets.MNIST` using the transform defined above
#### YOUR CODE STARTS HERE ####
train_dataset = datasets.MNIST(root = './data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root = './data', train=False, transform=transforms.ToTensor(), download=True)

#### YOUR CODE ENDS HERE ####

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 9912422/9912422 [00:00<00:00, 17895134.05it/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 28881/28881 [00:00<00:00, 502790.03it/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 1648877/1648877 [00:01<00:00, 1126951.81it/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100%|██████████| 4542/4542 [00:00<00:00, 4718981.61it/s]

Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw



In [ ]:
# create dataloaders for training and test datasets
# use a batch size of 32 and set shuffle=True for the training set
#### YOUR CODE STARTS HERE ####
train_dataloader = torch.utils.data.DataLoader(dataset = train_dataset, batch_size = 32, shuffle=True)
test_dataloader = torch.utils.data.DataLoader(dataset = test_dataset, batch_size = 32, shuffle=False)
#### YOUR CODE ENDS HERE ####


In [ ]:
len(train_dataset)

60000

### Network Definition

In [ ]:
# Convolutional neural network (Two convolutional layers)
class ConvolutionNet(nn.Module):
    def __init__(self, num_classes=10):
        super( ConvolutionNet, self).__init__()

        # define a sequential module with
        # 1. conv layer with input channel as 1, output channels as 16, kernel size of 5, stride of 1 and padding 0
        # 2. 2D BatchNorm of 16 features
        # 3. ReLU activation
        # 4. 2D MaxPool with kernel size of 2 and stride of 2

        #### YOUR CODE STARTS HERE ####
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(1,16,5,1,0),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )
        #### YOUR CODE ENDS HERE ####

        # define a sequential module with
        # 1. conv layer with input channel as 16, output channels as 8, kernel size of 7, stride of 1 and padding 2
        # 2. 2D BatchNorm of 8 features
        # 3. ReLU activation
        # 4. 2D MaxPool with kernel size of 2 and stride of 2

        #### YOUR CODE STARTS HERE ####
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(16,8,7,1,2),
            nn.BatchNorm2d(8),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        #### YOUR CODE ENDS HERE ####

        # define a linear(dense) layer with output features corresponding to the number of classes in the dataset

        #### YOUR CODE STARTS HERE ####
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(5*5*8, 10)
        )
        #### YOUR CODE ENDS HERE ####

    def forward(self, x):
        # Use the sequential convolution blocks defined above (conv_block1--> conv_block2-->fc) and
        # write the forward pass.

        #### YOUR CODE STARTS HERE ####
        output = self.fc(self.conv_block2(self.conv_block1(x)))
        #### YOUR CODE ENDS HERE ####
        return output


### Question 1

What is the size of parameter matrix corresponding to convolution layer of second sequential block ?

1. 8x16x7x7
2. 16x16x6x6
3. 16x8x5x5
4. 16x8x4x4


In [ ]:
#### YOUR CODE STARTS HERE ####
num_classes = 10
model = ConvolutionNet(num_classes).to(device)
model.conv_block2[0].weight.shape
#or
# params=list(model.parameters())
# params[4].shape
#### YOUR CODE ENDS HERE ####

torch.Size([8, 16, 7, 7])

### Training and Inference

In [ ]:
#define the model
#### YOUR CODE STARTS HERE ####
model = ConvolutionNet(num_classes).to(device)
#### YOUR CODE ENDS HERE ####


#define cross entropy loss and Adam optimizer using learning rate=learning_rate
#### YOUR CODE STARTS HERE ####
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
#### YOUR CODE ENDS HERE ####

# Train the model
total_step = len(train_dataloader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_dataloader):
       #### YOUR CODE STARTS HERE ####
        # send the image, target to the device
        images = images.to(device)
        labels = labels.to(device)

        # flush out the gradients stored in optimizer

        # pass the image to the model and assign the output to variable named output
        output = model(images)
        # calculate the loss (use cross entropy in pytorch)
        loss = criterion(output, labels)
        # do a backward pass
        optimizer.zero_grad()
        loss.backward()
        # update the weights
        optimizer.step()

       #### YOUR CODE ENDS HERE ####
        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

# Test the model
model.eval()  # eval mode (batchnorm uses moving mean/variance instead of mini-batch mean/variance)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_dataloader:
      ### YOUR CODE STARTS HERE ####
        # send the image, target to the device
        images = images.to(device)
        labels = labels.to(device)
        # pass the image to the model and assign the output to variable named output
        outputs = model(images)
      #### YOUR CODE ENDS HERE ####
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Test Accuracy of the model : {} %'.format(100 * correct / total))

Epoch [1/10], Step [100/1875], Loss: 0.5803
Epoch [1/10], Step [200/1875], Loss: 0.1076
Epoch [1/10], Step [300/1875], Loss: 0.1417
Epoch [1/10], Step [400/1875], Loss: 0.0692
Epoch [1/10], Step [500/1875], Loss: 0.2053
Epoch [1/10], Step [600/1875], Loss: 0.3224
Epoch [1/10], Step [700/1875], Loss: 0.0710
Epoch [1/10], Step [800/1875], Loss: 0.0048
Epoch [1/10], Step [900/1875], Loss: 0.1079
Epoch [1/10], Step [1000/1875], Loss: 0.0710
Epoch [1/10], Step [1100/1875], Loss: 0.0541
Epoch [1/10], Step [1200/1875], Loss: 0.0625
Epoch [1/10], Step [1300/1875], Loss: 0.1732
Epoch [1/10], Step [1400/1875], Loss: 0.0064
Epoch [1/10], Step [1500/1875], Loss: 0.0474
Epoch [1/10], Step [1600/1875], Loss: 0.0144
Epoch [1/10], Step [1700/1875], Loss: 0.1187
Epoch [1/10], Step [1800/1875], Loss: 0.1375
Epoch [2/10], Step [100/1875], Loss: 0.0699
Epoch [2/10], Step [200/1875], Loss: 0.0415
Epoch [2/10], Step [300/1875], Loss: 0.0472
Epoch [2/10], Step [400/1875], Loss: 0.0271
Epoch [2/10], Step [500

#### Question-2

Report the final test accuracy displayed above (If you are not getting the exact number shown in options, please report the closest number).
1. 84%
2. 76%
3. 99%
4. 57%


### Resnet with Squeeze and Excitation Block
In this question, you'll have to code ResNet with a Squeeze and Excitation block from scratch.

It's suggested you first briefly understand how the Squeeze and Excitation block works.

Sidenote: As this assignment is mainly focused on learning things, we don't focus on architecture design and hyper-parameter tuning. When you start using deep learning in real-world applications and competitions, hyper-parameter tuning plays a decent role!



In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torch.nn.functional as F
import timeit
import unittest

## Please DONOT remove these lines.
torch.manual_seed(2022)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(2022)

### Data Loading and Pre-processing

In [ ]:
# check availability of GPU and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# define a set of transforms for preparing the dataset
transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=8),
        transforms.RandomHorizontalFlip(), # flip the image horizontally (use pytorch random horizontal flip)
        transforms.ToTensor(), # convert the image to a pytorch tensor
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)) # normalise the images with mean and std of the dataset
        ])

# define transforms for the test data: Should they be same as the one used for train?
transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])

use_cuda = torch.cuda.is_available() # if you have acess to a GPU, enable it to speed the training

In [ ]:
# Load the CIFAR10 training, test datasets using `torchvision.datasets.CIFAR10` using the transform defined above
#### YOUR CODE STARTS HERE ####
train_dataset = datasets.CIFAR10("/data",train=True,transform= transforms.ToTensor(), download=True)
test_dataset = datasets.CIFAR10("/data",train=False,transform= transforms.ToTensor(), download=True)
#### YOUR CODE ENDS HERE ####

100%|██████████| 170498071/170498071 [00:05<00:00, 29133527.42it/s]


Extracting /data/cifar-10-python.tar.gz to /data
Files already downloaded and verified


In [ ]:
# create dataloaders for training and test datasets
# use a batch size of 32 and set shuffle=True for the training set
#### YOUR CODE STARTS HERE ####
train_dataloader = torch.utils.data.DataLoader(dataset = train_dataset, batch_size = 32, shuffle=True)
test_dataloader = torch.utils.data.DataLoader(dataset = test_dataset, batch_size = 32, shuffle=False)
#### YOUR CODE ENDS HERE ####

### Network Definition

In [ ]:
# Squeeze and excitation residual neural network (One convolutional layer and one resnet block)
class SEResNet(nn.Module):
    def __init__(self, num_classes=10):
        super( SEResNet, self).__init__()

        # define a sequential module named conv_block1 with
        # 1. conv layer with input channel as 3, output channels as 16, kernel size of 3, stride of 1 and padding 1
        # 2. 2D BatchNorm of 16 features
        # 3. ReLU activation
        # 4. 2D MaxPool with kernel size of 2 and stride of 2

        #### YOUR CODE STARTS HERE ####
        self.conv_block1 = nn.Sequential(
            nn.Conv2d(3,16,3,1,1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        #### YOUR CODE ENDS HERE ####

        # define a sequential module named conv_block2 with
        # 1. conv layer with input channel as 16, output channels as 16, kernel size of 5, stride of 1 and padding 1
        # 2. 2D BatchNorm of 64 features
        # 3. ReLU activation

        #### YOUR CODE STARTS HERE ####
        self.conv_block2 = nn.Sequential(
            nn.Conv2d(16,16,5,1,1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        #### YOUR CODE ENDS HERE ####

        # define a sequential module named conv_block3 with
        # 1. conv layer with input channel as 16, output channels as 16, kernel size of 5, stride of 1 and padding 3
        # 2. 2D BatchNorm of 16 features

        #### YOUR CODE STARTS HERE ####
        self.conv_block3 = nn.Sequential(
            nn.Conv2d(16,16,5,1,3),
            nn.BatchNorm2d(16)
        )
        #### YOUR CODE ENDS HERE ####

        #define a squeeze and excitation block with following requirements
        #1. the output of conv_block3 should be squeezed with average pooling while maintaining the number of channels
        #2. the excitation block should be a sequential module that passes the squeezed vector through a bottleneck of dimension 4
        #   and then then expand it back to its original size while using relu activation in the bottleneck layer and sigmoid in the expanded output(no bias should be used)
        #3. Define a single relu activation layer named relu

        #### YOUR CODE STARTS HERE ####
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(16,4),
            nn.ReLU(),
            nn.Linear(4,16),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU()
        #### YOUR CODE ENDS HERE ####

        # define a linear(dense) layer with output features corresponding to the number of classes in the dataset and input features as per the forward pass defined later

        #### YOUR CODE STARTS HERE ####
        self.fc = nn.Linear(16,10)
        #### YOUR CODE ENDS HERE ####

    def forward(self, x):
        # Use the blocks defined above to write the forward pass for the following squeeze and excitation resnet:
        #(input -> conv_block1 -> conv_block2 -> conv_block3 -> squeeze -> excite -> scale (ie scale the ouputs of conv_block3 with the corresponding excitations)->
        # skip connection(from conv_block1 to the scaled outputs) -> relu -> fc -> prediction

        #### YOUR CODE STARTS HERE ####
        output = self.fc(self.relu(self.excitation(self.squeeze(self.conv_block3(self.conv_block2(self.conv_block1(x)))))))
        print(output.shape)
        #### YOUR CODE ENDS HERE ####
        return output

### Training and Inference

In [ ]:
# Write the model definition, training and testing code exactly as before and replace model name by SEResNet
#### YOUR CODE STARTS HERE ####
model = SEResNet(num_classes).to(device)
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)
# Train the model
total_step = len(train_dataloader)
for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(train_dataloader):
       #### YOUR CODE STARTS HERE ####
        # send the image, target to the device
        images = images.to(device)
        labels = labels.to(device)

        # flush out the gradients stored in optimizer

        # pass the image to the model and assign the output to variable named output
        output = model(images)
        # calculate the loss (use cross entropy in pytorch)
        loss = criterion(output, labels)
        # do a backward pass
        optimizer.zero_grad()
        loss.backward()
        # update the weights
        optimizer.step()

       #### YOUR CODE ENDS HERE ####
        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}'
                   .format(epoch+1, num_epochs, i+1, total_step, loss.item()))

# Test the model
model.eval()  # eval mode (batchnorm uses moving mean/variance instead of mini-batch mean/variance)
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_dataloader:
      ### YOUR CODE STARTS HERE ####
        # send the image, target to the device
        images = images.to(device)
        labels = labels.to(device)
        # pass the image to the model and assign the output to variable named output
        outputs = model(images)
      #### YOUR CODE ENDS HERE ####
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Test Accuracy of the model : {} %'.format(100 * correct / total))


#### YOUR CODE ENDS HERE ####

RuntimeError: mat1 and mat2 shapes cannot be multiplied (512x1 and 16x4)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from tqdm import tqdm

# Device configuration (use GPU if available, otherwise use CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CIFAR-10 dataset
transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=8),
        transforms.RandomHorizontalFlip(), # flip the image horizontally (use pytorch random horizontal flip)
        transforms.ToTensor(), # convert the image to a pytorch tensor
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)) # normalise the images with mean and std of the dataset
        ])

# define transforms for the test data: Should they be same as the one used for train?
transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])

use_cuda = torch.cuda.is_available() # if you have acess to a GPU, enable it to speed the training
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)

# Squeeze-and-Excitation (SE) block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction_ratio=4):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction_ratio),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction_ratio, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        batch_size, channels, _, _ = x.size()
        z = self.squeeze(x).view(batch_size, channels)
        y = self.excitation(z).view(batch_size, channels, 1, 1)
        return x * y.expand_as(x)

# SENet model
class SENet(nn.Module):
    def __init__(self, num_classes=10, reduction_ratio=16):
        super(SENet, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(3,16,3,1,1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16,16,5,1,1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )
        self.conv3 =  nn.Sequential(
            nn.Conv2d(16,16,5,1,3),
            nn.BatchNorm2d(16)
        )
        self.relu = nn.ReLU(inplace=True)
        #self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        #self.se1 = SEBlock(64, reduction_ratio)
        #self.se2 = SEBlock(128, reduction_ratio)
        self.se3 = SEBlock(16, reduction_ratio)
        self.fc = nn.Linear(16*16*16, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.se3(x)
        x = x.view(x.size(0), -1)
        x = self.relu(x)
        x = self.fc(x)
        return x

# Initialize model, loss function, and optimizer
model = SENet(num_classes=10, reduction_ratio=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the model
def train_model(model, criterion, optimizer, num_epochs=1):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        print(f'Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}')

# Evaluating the model
def evaluate_model(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc='Evaluating'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_acc = correct / total
    print(f'Test Accuracy: {test_acc:.4f}')

# Train and evaluate the model
train_model(model, criterion, optimizer, num_epochs=10)
evaluate_model(model)


Files already downloaded and verified
Files already downloaded and verified


Epoch 1/10: 100%|██████████| 391/391 [01:05<00:00,  5.95it/s]


Train Loss: 1.6755 | Train Acc: 0.3850


Epoch 2/10: 100%|██████████| 391/391 [01:04<00:00,  6.10it/s]


Train Loss: 1.4103 | Train Acc: 0.4870


...

Train Loss: 1.0750 | Train Acc: 0.6166


Epoch 10/10: 100%|██████████| 391/391 [01:03<00:00,  6.13it/s]


Train Loss: 1.0542 | Train Acc: 0.6252


Evaluating: 100%|██████████| 100/100 [00:07<00:00, 13.65it/s]

Test Accuracy: 0.6322


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from tqdm import tqdm

# Device configuration (use GPU if available, otherwise use CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# CIFAR-10 dataset
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=100, shuffle=False, num_workers=2)

# Squeeze-and-Excitation (SE) block
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction_ratio=16):
        super(SEBlock, self).__init__()
        self.squeeze = nn.AdaptiveAvgPool2d(1)
        self.excitation = nn.Sequential(
            nn.Linear(in_channels, in_channels // reduction_ratio),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction_ratio, in_channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        batch_size, channels, _, _ = x.size()
        z = self.squeeze(x).view(batch_size, channels)
        y = self.excitation(z).view(batch_size, channels, 1, 1)
        return x * y.expand_as(x)

# SENet model
class SENet(nn.Module):
    def __init__(self, num_classes=10, reduction_ratio=16):
        super(SENet, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU(inplace=True)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.se1 = SEBlock(64, reduction_ratio)
        self.se2 = SEBlock(128, reduction_ratio)
        self.se3 = SEBlock(256, reduction_ratio)
        self.fc = nn.Linear(256 * 4 * 4, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.se1(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.se2(x)
        x = self.pool(x)

        x = self.conv3(x)
        x = self.relu(x)
        x = self.se3(x)
        x = self.pool(x)

        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Initialize model, loss function, and optimizer
model = SENet(num_classes=10, reduction_ratio=16).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training the model
def train_model(model, criterion, optimizer, num_epochs=10):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        correct = 0
        total = 0
        for inputs, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        print(f'Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}')

# Evaluating the model
def evaluate_model(model):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc='Evaluating'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_acc = correct / total
    print(f'Test Accuracy: {test_acc:.4f}')

# Train and evaluate the model
train_model(model, criterion, optimizer, num_epochs=10)
evaluate_model(model)


Files already downloaded and verified
Files already downloaded and verified


Epoch 1/10: 100%|██████████| 391/391 [06:54<00:00,  1.06s/it]


Train Loss: 1.6331 | Train Acc: 0.4052


Epoch 2/10: 100%|██████████| 391/391 [06:53<00:00,  1.06s/it]


Train Loss: 1.2852 | Train Acc: 0.5384


...

Train Loss: 0.6800 | Train Acc: 0.7658


Epoch 10/10: 100%|██████████| 391/391 [06:31<00:00,  1.00s/it]


Train Loss: 0.6415 | Train Acc: 0.7775


Evaluating: 100%|██████████| 100/100 [00:31<00:00,  3.16it/s]

Test Accuracy: 0.7749


#### Question-3

Report the final test accuracy displayed above (If you are not getting the exact number shown in options, please report the closest number).
1. 85%
2. 73%
3. 54%
4. 65%
